## Ce fichier IPython Notebook contient les étapes du processus de formation de notre modèle universitaire. Ce modèle Unibot a été spécialement conçu pour répondre aux questions des étudiants qui souhaitent rejoindre notre université. Les étapes de formation incluent la collecte des données pertinentes, le prétraitement du texte, la construction dynamique d'un vocabulaire, la création d'un modèle d'apprentissage automatique en utilisant les réseau de neurones et son entraînement. Une fois le modèle entraîné, il sera en mesure de répondre aux questions des étudiants potentiels en fournissant des informations sur les programmes d'études, les admissions, les ressources universitaires, et bien plus encore.

# **Importations**

In [ ]:
import json
import string
import nltk
nltk.download("punkt")
nltk.download("wordnet")
import random
import numpy as np
from nltk.stem import WordNetLemmatizer
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

# Prétraitement des **données**

In [ ]:
with open('intentsEnglish2.json') as file:
    data = json.load(file)

# initialisation de lemmatizer pour obtenir la base des mots
lemmatizer = WordNetLemmatizer()
# création des listes
words = []
classes = []
doc_X = []
doc_y = []
# parcourir avec une boucle pour toutes les intentions
# tokéniser chaque pattern et ajouter les tokens à la liste words, les patterns et
# le tag associé à l'intention sont ajoutés aux listes correspondantes
for intent in data["intents"]:
    for pattern in intent["patterns"]:
        tokens = nltk.word_tokenize(pattern)
        # Filter out words with length <= 2
        filtered_tokens = [token for token in tokens if len(token) > 2]
        words.extend(filtered_tokens)

        doc_X.append(pattern)
        doc_y.append(intent["tag"])

    # ajouter le tag unique aux classes
    if intent["tag"] not in classes:
        classes.append(intent["tag"])
# lemmatiser tous les mots du vocabulaire et les convertir en minuscule
# si les mots n'apparaissent pas dans la ponctuation
words = [lemmatizer.lemmatize(word.lower()) for word in words if word not in string.punctuation]
# trier le vocabulaire et les classes par ordre alphabétique et prendre le
# set pour s'assurer qu'il n'y a pas de doublons
words = sorted(set(words))
classes = sorted(set(classes))

## visualisation des lists pour une meilleur **comprehension**

In [ ]:
print(words)
print(classes)

["'re", 'able', 'about', 'abroad', 'academic', 'accepted', 'access', 'accommodation', 'accéder', 'across', 'activity', 'actually', 'additional', 'address', 'administration', 'administrative', 'admission', 'admitted', 'adopting', 'advantage', 'advisor', 'afraid', 'after', 'afternoon', 'again', 'agreement', 'aid', 'aim', 'alumnus', 'aménagé', 'and', 'another', 'antecedent', 'any', 'applies', 'apply', 'appreciate', 'approval', 'architecturally', 'architecture', 'are', 'area', 'arrangement', 'aspect', 'assessed', 'assist', 'assistance', 'associated', 'association', 'attend', 'attraction', 'aux', 'availability', 'available', 'aware', 'bac', 'bachelor', 'banking', 'beauty', 'benefit', 'besides', 'between', 'book', 'both', 'break', 'bring', 'building', 'bursary', 'but', 'calendar', 'campus', 'can', 'candidate', 'career', 'catalogue', 'center', 'certification', 'choose', 'chosen', 'clarify', 'club', 'collaborate', 'commercialization', 'communication', 'competency', 'complete', 'concept', 'conc

In [ ]:
print(doc_X)

['I want to know your name', 'Can you please tell me your name?', 'Who are you?', 'What are you?', 'Who you are?', 'who am I speaking to?', 'Could you help me?', 'give me a hand please', 'Can you help?', 'can I get your help?', 'I need assistance, can you guide me?', "I'm facing an issue, could you lend a hand?", 'hi', 'hello', 'hey', 'good morning', 'good afternoon', 'good evening', 'thank you', 'thanks a lot', 'thank you so much', 'I appreciate your help', "I'm grateful", 'thank you very much', 'Can you provide more information about the education program?', 'How does this education program work?', 'Does ESTM offer other types of programs besides DUT?', 'What is the semester structure in the DUT program?', 'What evaluation methods are used in this program?', 'How are the teaching programs structured?', 'What are the major options for students at EST?', "I'm interested in EST's majors", 'Can I get information about specific majors?', 'What study fields and majors can be chosen at EST?

In [ ]:
print(doc_y)

['about', 'about', 'about', 'about', 'about', 'about', 'help', 'help', 'help', 'help', 'help', 'help', 'salutation', 'salutation', 'salutation', 'salutation', 'salutation', 'salutation', 'thanksgiving', 'thanksgiving', 'thanksgiving', 'thanksgiving', 'thanksgiving', 'thanksgiving', 'education', 'education', 'education', 'education', 'education', 'education', 'majors', 'majors', 'majors', 'majors', 'majors', 'majors', 'majors', 'Application Process', 'Application Process', 'Application Process', 'Application Process', 'Application Process', 'Application Process', 'Admission Requirements', 'Admission Requirements', 'Admission Requirements', 'Admission Requirements', 'Admission Requirements', 'Admission Requirements', 'TM', 'TM', 'TM', 'TM', 'TM', 'TM', 'FBA', 'FBA', 'FBA', 'FBA', 'FBA', 'FBA', 'TCC', 'TCC', 'TCC', 'TCC', 'TCC', 'TCC', 'Catalogue-Umi', 'Catalogue-Umi', 'Catalogue-Umi', 'Catalogue-Umi', 'Catalogue-Umi', 'Catalogue-Umi', 'Partenaires', 'Partenaires', 'Partenaires', 'Partena

# Préparation des données d'**entraînement**

In [ ]:
# liste pour les données d'entraînement
training = []
out_empty = [0] * len(classes)
# création du modèle d'ensemble de mots
for idx, doc in enumerate(doc_X):
    bow = []
    text = lemmatizer.lemmatize(doc.lower())
    for word in words:
        bow.append(1) if word in text else bow.append(0)
    print(words)
    print(doc)
    print(bow)
    # marque l'index de la classe à laquelle le pattern atguel est associé à
    output_row = list(out_empty)
    output_row[classes.index(doc_y[idx])] = 1
    # ajoute le one hot encoded BoW et les classes associées à la liste training
    training.append([bow, output_row])
# mélanger les données et les convertir en array
random.shuffle(training)
training = np.array(training, dtype=object)
# séparer les features et les labels target
train_X = np.array(list(training[:, 0]))
train_y = np.array(list(training[:, 1]))



["'re", 'able', 'about', 'abroad', 'academic', 'accepted', 'access', 'accommodation', 'accéder', 'across', 'activity', 'actually', 'additional', 'address', 'administration', 'administrative', 'admission', 'admitted', 'adopting', 'advantage', 'advisor', 'afraid', 'after', 'afternoon', 'again', 'agreement', 'aid', 'aim', 'alumnus', 'aménagé', 'and', 'another', 'antecedent', 'any', 'applies', 'apply', 'appreciate', 'approval', 'architecturally', 'architecture', 'are', 'area', 'arrangement', 'aspect', 'assessed', 'assist', 'assistance', 'associated', 'association', 'attend', 'attraction', 'aux', 'availability', 'available', 'aware', 'bac', 'bachelor', 'banking', 'beauty', 'benefit', 'besides', 'between', 'book', 'both', 'break', 'bring', 'building', 'bursary', 'but', 'calendar', 'campus', 'can', 'candidate', 'career', 'catalogue', 'center', 'certification', 'choose', 'chosen', 'clarify', 'club', 'collaborate', 'commercialization', 'communication', 'competency', 'complete', 'concept', 'conc

# définition de quelques **paramètres**

In [ ]:
input_shape = (len(train_X[0]),)
output_shape = len(train_y[0])

## **Architecture** du Modèle de **Deep** **Learning**

In [ ]:
#modèle Deep Learning
model = Sequential()
model.add(Dense(128, input_shape=input_shape, activation="relu"))
model.add(Dropout(0.5))
model.add(Dense(64, activation="relu"))
model.add(Dropout(0.3))
model.add(Dense(output_shape, activation = "softmax"))
# Create an instance of the Adam optimizer
adam = Adam(learning_rate=0.01)

# Compile your model
model.compile(loss='categorical_crossentropy', optimizer=adam, metrics=["accuracy"])
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 128)               59264     
                                                                 
 dropout (Dropout)           (None, 128)               0         
                                                                 
 dense_1 (Dense)             (None, 64)                8256      
                                                                 
 dropout_1 (Dropout)         (None, 64)                0         
                                                                 
 dense_2 (Dense)             (None, 34)                2210      
                                                                 
Total params: 69730 (272.38 KB)
Trainable params: 69730 (272.38 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


# ***Entraînement*** du **modèle**

In [ ]:
model.fit(x=train_X, y=train_y, epochs=200, verbose=1)

Epoch 1/200
7/7 [==============================] - 1s 6ms/step - loss: 3.5242 - accuracy: 0.0561
Epoch 2/200
7/7 [==============================] - 0s 5ms/step - loss: 3.1238 - accuracy: 0.2296
Epoch 3/200
7/7 [==============================] - 0s 4ms/step - loss: 2.6144 - accuracy: 0.3520
Epoch 4/200
7/7 [==============================] - 0s 4ms/step - loss: 2.1103 - accuracy: 0.4337
Epoch 5/200
7/7 [==============================] - 0s 5ms/step - loss: 1.7005 - accuracy: 0.5561
Epoch 6/200
7/7 [==============================] - 0s 8ms/step - loss: 1.2585 - accuracy: 0.6888
Epoch 7/200
7/7 [==============================] - 0s 7ms/step - loss: 0.9588 - accuracy: 0.7806
Epoch 8/200
7/7 [==============================] - 0s 7ms/step - loss: 0.8368 - accuracy: 0.7602
Epoch 9/200
7/7 [==============================] - 0s 6ms/step - loss: 0.6155 - accuracy: 0.8112
Epoch 10/200
7/7 [==============================] - 0s 7ms/step - loss: 0.5502 - accuracy: 0.8367
Epoch 11/200
7/7 [===========

# Fonctions de prédiction et de réponse pour le **déploiement** du modèle

In [ ]:
def clean_text(text):
  tokens = nltk.word_tokenize(text)
  tokens = [lemmatizer.lemmatize(word) for word in tokens]
  return tokens

def bag_of_words(text, vocab):
  tokens = clean_text(text)
  bow = [0] * len(vocab)
  for w in tokens:
    for idx, word in enumerate(vocab):
      if word == w:
        bow[idx] = 1
  return np.array(bow)

def pred_class(text, vocab, labels):
  bow = bag_of_words(text, vocab)
  result = model.predict(np.array([bow]))[0]
  thresh = 0.2
  y_pred = [[idx, res] for idx, res in enumerate(result) if res > thresh]
  y_pred.sort(key=lambda x: x[1], reverse=True)
  return_list = []
  for r in y_pred:
    return_list.append(labels[r[0]])
  return return_list

def get_response(intents_list, intents_json):
  tag = intents_list[0]
  list_of_intents = intents_json["intents"]
  for i in list_of_intents:
    if i["tag"] == tag:
      result = random.choice(i["responses"])
      break
  return result




#***lancement*** du **chatbot**

In [ ]:
while True:
    message = input("")

    if message.lower() == "quit" or message.lower() == "stop":
        print("Have a wonderful day! if you have any further questions or need assistance, please don't hesitate to reach out.")
        break

    intents = pred_class(message, words, classes)
    max_pred_value = model.predict(np.array([bag_of_words(message, words)]))[0].max()

    if max_pred_value < 0.4:
        print("Chatbot: I'm sorry, I think this may be out of context, Let's bring the discussion back to the realm of university-related subjects.")
    else:
        result = get_response(intents, data)
        print("Chatbot:", result)

heyuuu
1/1 [==============================] - 0s 33ms/step
Chatbot: Welcome! How can I assist you today?
i want ti kno yur name
1/1 [==============================] - 0s 20ms/step
Chatbot: I'm EduBot, your bot assistant through your univesity adventure
cool, thanks
1/1 [==============================] - 0s 33ms/step
Chatbot: I'm here for that! Don't hesitate to reach out if you need more information.
what is the capital of spain?
1/1 [==============================] - 0s 21ms/step
Chatbot: I'm sorry, I don't understand. Could you please rephrase your question?
stop
Bonne journée ! N'hésitez pas à revenir si vous avez des questions.
